## Tomographic reconstruction 

This notebook is used for obtaining a 3D volume from a series of projections, usually obtained from a ptycho-tomography scan. 

The reconstruction process includes: 
- Uploading the data
- Phase unwrapping of data
- Doing vertical motion correction of the data
- Doing horizontal motion correction of the data
- 3D reconstructing 

In [ ]:
# original author: Oriol
# Danica rewrote the code using the nomenclature from our repository.
# The functionality might be slightly different from original but the pipeline was approximately untouched.
#------------------------------------------------------------------------------------------------------------

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import logging
import warnings
warnings.filterwarnings('ignore')
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)
import matplotlib.pyplot as plt
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
%matplotlib inline

from io_module.Imports import ImportData
from viewer.OpenViewer import OpenViewer
from CT_reconstruction.TomoRecon import get_volume
from utilities.Unwrap import Unwrap
from utilities.RemoveProjections import remove_stripes_oriol
from alignment.Alignment import VerticalAlignmentCrossCorrelation, HorizontalAlignmentCrossCorrelation, COMAlignment

In [ ]:
# This file should point to all the projections (already retrieved and ready to go). It's a nexus file (?). 
from config.paths import pollen_filepath, pollen_data_key, pollen_angle_key

ptytomofile = pollen_filepath
ptytomofile



In [ ]:
data = ImportData(ptytomofile, data_key=pollen_data_key, angle_key = pollen_angle_key)
projections_raw, angles, probes = data.get_projections_raw(), data.get_angles(), data.get_probes()

In [ ]:
OpenViewer(projections_raw)

In [ ]:
projections_in = np.copy(projections_raw) # copy the raw projections so that we don't lose them
projections_in = np.amin(projections_in) - projections_in # ???

In [ ]:
Unwrap(projections_in, True) #this takes 5 mins on Danica's laptop with parallel true



In [ ]:
# Save it as a .npy file
np.save("projections_in.npy", projections_in)

In [ ]:
# Load the saved array
projections_in = np.load(r"C:\Users\zvm34551\Coding_environment\PtychographyAlignment\data\experimental\data\projections_in.npy")

In [ ]:
OpenViewer(projections_in)


In [ ]:
def crop(projections, crop_window):
    projections_crop = projections[:,crop_window[0]:crop_window[1],crop_window[2]:crop_window[3]]
    return projections_crop

In [ ]:
crop_window = [700,1400,1000,2300] # [y1, y2, x1, x2] where y is vertical and x is horizontal
projections_crop = crop(projections_in, crop_window)


In [ ]:
projections_crop.shape

In [ ]:
OpenViewer(projections_crop)

In [ ]:


# Reconstruct including all the "dead" frames, i.e. those acquired during beam dumping
volume_gridrec_beamDumping = get_volume(projections_crop, angles, centre=None, pad=200, algorithm='GRIDREC', iterations=1)


In [ ]:
OpenViewer(volume_gridrec_beamDumping)

## Corrections / Normalise

In [ ]:
# show an example of the stripes caused by the beam dumping
#for i in range(projections_crop.shape[0]):
#    projections_crop[i,:,:] = projections_crop[i,:,:]/np.mean(projections_crop[i,1:10,1:10])

plt.figure(figsize=[5,3])
plt.plot(np.abs(np.sum(projections_crop[:,projections_crop.shape[1]//2,:],axis = 1)))
plt.title('Periodicity of stripes caused by beam dumping')

In [ ]:
# remove stripes
projections_reduced, angles_reduced, good_projs = remove_stripes_oriol(projections_crop, angles, threshold=0.4)


In [ ]:
print(projections_crop)

In [ ]:
np.save("good_projs.npy", good_projs)

In [ ]:
print(projections_reduced.shape)

In [ ]:
# Save it as a .npy file
np.save("projections_reduced.npy", projections_reduced)

In [ ]:
print(projections_reduced)

In [ ]:
angles_reduced

In [ ]:
np.save("angles_reduced.npy", angles_reduced)

In [ ]:
print(good_projs)

In [ ]:
OpenViewer(projections_reduced)

In [ ]:
# this will just show the new sinogram with 0s where we have removed the original projections
# It allows us to visually assess how many projections we're removing
sino_number = 150
new_sinogram = np.zeros_like(projections_crop[:,sino_number,:])
new_sinogram[good_projs[0],:] = projections_reduced[:,sino_number,:]
plt.figure()
plt.imshow(new_sinogram)

In [ ]:





# plot after fixing stripes
OpenViewer(projections_reduced)


In [ ]:


plt.figure(figsize=[5,3])
plt.plot(np.abs(np.sum(projections_reduced[:,projections_reduced.shape[1]//2,:],axis = 1)))
plt.title('Periodicity of stripes caused by beam dumping (after correction)')

In [ ]:
# reconstruct now
volume_gridrec_reduced = get_volume(projections_reduced, angles_reduced, centre=None, pad=200, algorithm='GRIDREC', iterations=1)


In [ ]:
OpenViewer(volume_gridrec_reduced)

## In-depth alignment

Vertical alignment

In [ ]:
projections_recon_align = np.copy(projections_reduced) # copy projections reduced so that we don't lose them

In [ ]:
print(projections_recon_align.shape)

In [ ]:
# Do vertical motion correction

vacc = VerticalAlignmentCrossCorrelation(projections_recon_align)
projections_aligned_vert, vertical_shifts = vacc.projections_aligned, vacc.vertical_shifts # do the vertical alignment 

print('Before correction, shape: ', projections_reduced.shape)
print('After correction, shape: ', projections_aligned_vert.shape)

In [ ]:
# Save it as a .npy file
np.save("projections_aligned_vert.npy", projections_aligned_vert)

In [ ]:
OpenViewer(projections_aligned_vert)

In [ ]:


# plot first and last projections before and after correction
plt.figure(figsize=[10,4])
plt.subplot(2,2,1), plt.imshow(projections_reduced[0,:,:]), plt.title('Proj 0 before') # before
plt.subplot(2,2,2), plt.imshow(projections_aligned_vert[0,:,:]), plt.title('Proj 0 after')# after
plt.subplot(2,2,3), plt.imshow(projections_reduced[-1,:,:]), plt.title('Last proj before')# before
plt.subplot(2,2,4), plt.imshow(projections_aligned_vert[-1,:,:]), plt.title('Last proj after')# after

plt.figure(figsize=[4,4])
plt.imshow(projections_aligned_vert[:,200,:])

In [ ]:
# reconstruct now
volume_gridrec_vert = get_volume(projections_aligned_vert, angles_reduced, centre=None, pad=200, algorithm='GRIDREC', iterations=1)

In [ ]:
OpenViewer(volume_gridrec_vert)


In [ ]:
# Pad the projections because we'll start using np.roll
print(projections_aligned_vert.shape)
projections_aligned_vert = np.pad(projections_aligned_vert, ((0,0),(0,0),(200,200)), mode='edge') #(projections_aligned_vert, (100,0)) 
plt.imshow(projections_aligned_vert[:,200,:])
print(projections_aligned_vert.shape)

In [ ]:
# Save it as a .npy file
np.save("projections_aligned_vert.npy", projections_aligned_vert)

In [ ]:
# Save it as a .npy file
projections_aligned_vert = np.load("projections_aligned_vert.npy")

In [ ]:
###-----------------------------------------------------------------------------------------------------------------
# vertical correction 
# Manual trial-and-error step to fix any large misalignments
proj1 = 500
misaligned_proj = 21 # with respect to proj1

correction_value = 10
misaligned_projection = proj1+misaligned_proj
projections_aligned_vert_fix = np.copy(projections_aligned_vert)
projections_aligned_vert_fix[misaligned_projection:-1,:,:] = np.roll(projections_aligned_vert[misaligned_projection:-1,:,:],correction_value,axis = 2)
plt.figure(figsize=[10,10])
plt.subplot(1,2,1), plt.imshow(projections_aligned_vert_fix[:,projections_aligned_vert_fix.shape[1]//2,:]), plt.title('Manually corrected')
plt.subplot(1,2,2), plt.imshow(projections_aligned_vert[:,projections_aligned_vert.shape[1]//2,:]), plt.title('Uncorrected')
plt.show()

###-----------------------------------------------------------------------------------------------------------------

# Manual trial-and-error step to fix any large misalignments
proj1 = 700
misaligned_proj = 32 # with respect to proj1
correction_value = 790

misaligned_projection = proj1+misaligned_proj

projections_aligned_vert_fix2 = np.copy(projections_aligned_vert_fix)
projections_aligned_vert_fix2[misaligned_projection,:,:] = np.roll(projections_aligned_vert_fix[misaligned_projection,:,:],correction_value,axis = 1)

plt.figure(figsize=[10,10])
plt.subplot(1,2,1), plt.imshow(projections_aligned_vert_fix2[:,projections_aligned_vert_fix2.shape[1]//2,:]), plt.title('Manually corrected')
plt.subplot(1,2,2), plt.imshow(projections_aligned_vert[:,projections_aligned_vert_fix.shape[1]//2,:]), plt.title('Uncorrected')
plt.show()


In [ ]:
# Save it as a .npy file
np.save("projections_aligned_vert_fix2.npy", projections_aligned_vert_fix2)

In [ ]:
correction_rough = HorizontalAlignmentCrossCorrelation(projections_aligned_vert_fix2).correction_rough


In [ ]:
OpenViewer(correction_rough)

In [ ]:
volume_gridrec_aligned = get_volume(correction_rough, angles_reduced, centre=None, pad=200, algorithm='GRIDREC', iterations=1)



In [ ]:
OpenViewer(volume_gridrec_aligned)

In [ ]:
# calculate the rough overall translation between 0 and 180 degrees by using cross-correlation between projection 1 
# and the last projection, flipped
# shifty = register_translation(projections_aligned_vert_fix[0,:,:],np.fliplr(projections_aligned_vert_fix[-1,:,:]),upsample_factor=100)
# print('The shift between 0 deg and 180 deg is roughly ' + str(shifty[0][0]) + ' in the vertical direction and '
#                     + str(shifty[0][1]) + ' in the horizontal direction.')

# Calculate the centre of mass (COM) for each projection of the scan. The COM should follow a sinusoidal motion.
# Then fit a sinusoid and shift the projections as required to fit that sinusoid. 
projections_fitted = COMAlignment(projections_aligned_vert_fix, correction_rough, angles_reduced).projections_fitted



In [ ]:
# Reconstruct 
volume_gridrec_aligned2 = get_volume(projections_fitted, angles_reduced, centre=None, pad=200, algorithm='GRIDREC', iterations=1)


In [ ]:
OpenViewer(volume_gridrec_aligned2[:,:,200:-50])

In [ ]:

# Compare reconstructions before and after alignment
OpenViewer(volume_gridrec_aligned[:,:,200:-50])


## Save the results

In [ ]:
import os 
os.getcwd()

In [ ]:
# save the volume -- I haven't actually tested this
import h5py
f = h5py.File('/dls/staging/dls/i13-1/data/2024/mg34773-2/processing/tomo/386068/tomo_gridrec_aligned.h5','w')
f.create_dataset('data', data=volume_gridrec_aligned)
f.close()